# 20 - Final Hallucination and Error Analysis

Create report-ready hallucination/error-analysis CSV files for Base RAG and Fine-tuned RAG outputs. This notebook uses deterministic retrieval/citation/answer-overlap signals from `evaluation_qa.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

reports_dir = DRIVE_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.evaluation_qa import evaluate_generation_predictions
from src.error_analysis import make_error_analysis

systems = [
    {
        'name': 'gemma_base_rag',
        'predictions': DRIVE_ROOT / 'outputs/generation_eval/base_rag_predictions_v1.csv',
        'eval': DRIVE_ROOT / 'outputs/generation_eval/base_rag_eval_v1.csv',
        'summary': DRIVE_ROOT / 'outputs/generation_eval/base_rag_summary_v1.json',
        'errors': reports_dir / 'error_analysis_gemma_base_rag.csv',
    },
    {
        'name': 'gemma_finetuned_rag',
        'predictions': DRIVE_ROOT / 'outputs/generation_eval/finetuned_rag_predictions_v1.csv',
        'eval': DRIVE_ROOT / 'outputs/generation_eval/finetuned_rag_eval_v1.csv',
        'summary': DRIVE_ROOT / 'outputs/generation_eval/finetuned_rag_summary_v1.json',
        'errors': reports_dir / 'error_analysis_gemma_finetuned_rag.csv',
    },
    {
        'name': 'qwen3_32b_base_rag',
        'predictions': DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_predictions_v1.csv',
        'eval': DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_eval_v1.csv',
        'summary': DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_summary_v1.json',
        'errors': reports_dir / 'error_analysis_qwen3_32b_base_rag.csv',
    },
    {
        'name': 'qwen3_32b_finetuned_rag',
        'predictions': DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_finetuned_rag_predictions_v1.csv',
        'eval': DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_finetuned_rag_eval_v1.csv',
        'summary': DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_finetuned_rag_summary_v1.json',
        'errors': reports_dir / 'error_analysis_qwen3_32b_finetuned_rag.csv',
    },
]

rows = []
for system in systems:
    if not system['predictions'].exists():
        rows.append({'system': system['name'], 'status': 'missing_predictions'})
        continue
    summary = evaluate_generation_predictions(
        predictions_csv=system['predictions'],
        output_eval_csv=system['eval'],
        output_summary_json=system['summary'],
    )
    errors = make_error_analysis(
        eval_csv=system['eval'],
        output_csv=system['errors'],
        per_type_limit=30,
    )
    row = {'system': system['name'], 'status': 'ok', 'error_csv': str(system['errors'])}
    row.update(summary['metrics'])
    row.update({f"error_{k}": v for k, v in summary.get('error_type_counts', {}).items()})
    rows.append(row)

hallucination_summary = pd.DataFrame(rows)
hallucination_summary.to_csv(reports_dir / 'final_hallucination_error_summary.csv', index=False, encoding='utf-8-sig')
hallucination_summary

In [ ]:
md_lines = ['# Final Hallucination and Error Analysis\n']
md_lines.append('The project uses deterministic hallucination/error proxies based on retrieval availability, citation presence, gold citation match, grounded citation score, and answer-overlap. Error types are: retrieval_miss, missing_citation, wrong_or_unsupported_citation, low_answer_overlap, and acceptable_automatic.\n')
md_lines.append(hallucination_summary.to_markdown(index=False))
(reports_dir / 'final_hallucination_error_summary.md').write_text('\n\n'.join(md_lines), encoding='utf-8')
reports_dir / 'final_hallucination_error_summary.md'